# Libraries

In [ ]:
# !pip install ipympl
%matplotlib widget # makes sure plots are saved in folder and not displayed in notebook

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn import svm
from sklearn.feature_selection import RFE
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, make_scorer

# Confusion Matrix Function

In [ ]:
def cm_plot(cm_scores, acc_scores, model, total=False):
    if total:
        # Compute the total confusion matrix
        cm_value = np.sum(cm_scores, axis=0)
        title = f'Confusion Matrix - Total F1-score - {model}'
    else:
        # Retrieve the confusion matrix with the highest F1-score
        cm_value = cm_scores[np.argmax(acc_scores)]
        title = f'Confusion Matrix - Highest F1-score - {model}'

    classes = ['Low mental workload', 'High mental workload']


    plt.figure(figsize=(8, 6))
    plt.imshow(cm_value, interpolation='nearest', cmap='coolwarm')
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')

    # Add the values to the confusion matrix plot
    thresh = cm_value.max() / 2.0
    for i in range(cm_value.shape[0]):
        for j in range(cm_value.shape[1]):
            plt.text(j, i, format(cm_value[i, j], 'd'), ha='center', va='center',
                     color='white' if cm_value[i, j] > thresh else 'black')

    plt.tight_layout()
    plt.show()
    output_file = f'/Users/basverkennis/Desktop/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/confusion_matrices/cmplot_{model}.png'
    # output_file = f'/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/confusion_matrices/cmplot_{model}.png'
    plt.savefig(output_file, bbox_inches='tight')
    plt.close()

# Fronto-Parietal Phase Locking Value (PLV) Connectivity function
### Retrieving the average and standardized version of the Fronto-Parietal Phase Locking Value (PLV) Connectivity features

In [ ]:
def fronto_parietal_averagePLV(data):
    '''
    Computes the mean connectivity between frontal and parietal channels bidirectionally.
    
    Parameters:
        data: Connectivity data for each channel.
    
    Returns:
        numpy.ndarray: (average) fronto-parietal connectivity values for each channel and for each frequency band (alpha, beta, theta).
    '''
    
    # Alpha mean PLV values
    F7_A = np.mean(data[4:9])
    F3_A = np.mean(data[12:17])
    Fz_A = np.mean(data[19:24])
    F4_A = np.mean(data[25:30])
    F8_A = np.mean(data[30:35])
    P3_A = np.mean(np.concatenate((data[4:5], data[12:13], data[19:20], data[25:26], data[30:31])))
    Pz_A = np.mean(np.concatenate((data[5:6], data[13:14], data[20:21], data[26:27], data[31:32])))
    P4_A = np.mean(np.concatenate((data[6:7], data[14:15], data[21:22], data[27:28], data[32:33])))
    PO7_A = np.mean(np.concatenate((data[7:8], data[15:16], data[22:23], data[28:29], data[33:34])))
    PO8_A = np.mean(np.concatenate((data[8:9], data[16:17], data[23:24], data[29:30], data[34:35])))
    
    # Beta mean PLV values
    F7_B = np.mean(data[4+45:9+45])
    F3_B = np.mean(data[12+45:17+45])
    Fz_B = np.mean(data[19+45:24+45])
    F4_B = np.mean(data[25+45:30+45])
    F8_B = np.mean(data[30+45:35+45])
    P3_B = np.mean(np.concatenate((data[4+45:5+45], data[12+45:13+45], data[19+45:20+45], data[25+45:26+45], data[30+45:31+45])))
    Pz_B = np.mean(np.concatenate((data[5+45:6+45], data[13+45:14+45], data[20+45:21+45], data[26+45:27+45], data[31+45:32+45])))
    P4_B = np.mean(np.concatenate((data[6+45:7+45], data[14+45:15+45], data[21+45:22+45], data[27+45:28+45], data[32+45:33+45])))
    PO7_B = np.mean(np.concatenate((data[7+45:8+45], data[15+45:16+45], data[22+45:23+45], data[28+45:29+45], data[33+45:34+45])))
    PO8_B = np.mean(np.concatenate((data[8+45:9+45], data[16+45:17+45], data[23+45:24+45], data[29+45:30+45], data[34+45:35+45])))
    
    # Theta mean PLV values
    F7_T = np.mean(data[4+90:9+90])
    F3_T = np.mean(data[12+90:17+90])
    Fz_T = np.mean(data[19+90:24+90])
    F4_T = np.mean(data[25+90:30+90])
    F8_T = np.mean(data[30+90:35+90])
    P3_T = np.mean(np.concatenate((data[4+90:5+90], data[12+90:13+90], data[19+90:20+90], data[25+90:26+90], data[30+90:31+90])))
    Pz_T = np.mean(np.concatenate((data[5+90:6+90], data[13+90:14+90], data[20+90:21+90], data[26+90:27+90], data[31+90:32+90])))
    P4_T = np.mean(np.concatenate((data[6+90:7+90], data[14+90:15+90], data[21+90:22+90], data[27+90:28+90], data[32+90:33+90])))
    PO7_T = np.mean(np.concatenate((data[7+90:8+90], data[15+90:16+90], data[22+90:23+90], data[28+90:29+90], data[33+90:34+90])))
    PO8_T = np.mean(np.concatenate((data[8+90:9+90], data[16+90:17+90], data[23+90:24+90], data[29+90:30+90], data[34+90:35+90])))
    
    min_value = np.min([F7_A, F3_A, Fz_A, F4_A, F8_A, P3_A, Pz_A, P4_A, PO7_A, PO8_A, F7_B, F3_B, Fz_B, F4_B, F8_B, P3_B, Pz_B, P4_B, PO7_B, PO8_B, F7_T, F3_T, Fz_T, F4_T, F8_T, P3_T, Pz_T, P4_T, PO7_T, PO8_T])
    max_value = np.max([F7_A, F3_A, Fz_A, F4_A, F8_A, P3_A, Pz_A, P4_A, PO7_A, PO8_A, F7_B, F3_B, Fz_B, F4_B, F8_B, P3_B, Pz_B, P4_B, PO7_B, PO8_B, F7_T, F3_T, Fz_T, F4_T, F8_T, P3_T, Pz_T, P4_T, PO7_T, PO8_T])
    
    PLVs_alpha = [(x - min_value) / (max_value - min_value) * 2 - 1 for x in [F7_A, F3_A, Fz_A, F4_A, F8_A, P3_A, Pz_A, P4_A, PO7_A, PO8_A]]    
    PLVs_theta = [(x - min_value) / (max_value - min_value) * 2 - 1 for x in [F7_B, F3_B, Fz_B, F4_B, F8_B, P3_B, Pz_B, P4_B, PO7_B, PO8_B]]
    PLVs_beta = [(x - min_value) / (max_value - min_value) * 2 - 1 for x in [F7_T, F3_T, Fz_T, F4_T, F8_T, P3_T, Pz_T, P4_T, PO7_T, PO8_T]]
    
    data_x = np.concatenate((PLVs_alpha, PLVs_beta, PLVs_theta))

    return data_x


# Baseline Model - RFE on 42 spectral features and SVM

In [ ]:
# Loading data 

directory = '/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'
concatenated_X, concatenated_Y = [], []
for subdir in sorted(os.listdir(directory)):
    for file in sorted(os.listdir(os.path.join(directory, subdir))):
        # print(directory.split('/')[-1], subdir, file)
        data = np.load(os.path.join(directory, subdir, file))
        X = data['X'][135:] # extract all spectral features
        Y = data['Y']
        concatenated_X.append(X)
        concatenated_Y.append(Y)

concatenated_Y = np.concatenate(concatenated_Y)
    

In [ ]:
# Recursive Feature Selection and Support Vector Machine - Binary Classification (RFE + SVM)

X = concatenated_X
Y = concatenated_Y
classifier = svm.SVC(random_state=42)
parameters = {'C': [0.1, 0.95, 0.99, 10], 'kernel': ['linear', 'rbf', 'sigmoid', 'poly'], 'gamma': [0.0001, 0.01, 0.95, 1, 10, 25]}
scoring = make_scorer(accuracy_score)
num_folds = 10
num_features = 8
kf = KFold(n_splits=num_folds, shuffle=False)
f1_scores, accuracy_scores, precision_scores, recall_scores, cm_scores, best_parameters_list = [],[],[],[],[],[]

for k, (train_index, test_index) in enumerate(kf.split(X), start=1):
    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    Y_train, Y_test = Y[train_index], Y[test_index]
    
    # Perform feature selection using RFE with LinearSVC as the estimator
    estimator = svm.LinearSVC(max_iter=1000000)
    selector = RFE(estimator, n_features_to_select=num_features)
    selector.fit(X_train, Y_train)
    X_train_selected = selector.transform(X_train)
    X_test_selected = selector.transform(X_test)
    
    selected_feature_indices = selector.support_
    selected_feature_ranking = selector.ranking_

    grid_search = GridSearchCV(classifier, parameters, scoring=scoring)
    grid_search.fit(X_train_selected, Y_train)

    best_parameters = grid_search.best_params_

    classifier = svm.SVC(kernel=best_parameters['kernel'], C=best_parameters['C'], gamma=best_parameters['gamma'])
    classifier.fit(X_train_selected, Y_train)

    Y_pred = classifier.predict(X_test_selected)

    best_parameters_list.append(best_parameters)
    f1_scores.append(f1_score(Y_test, Y_pred, zero_division=1))
    accuracy_scores.append(accuracy_score(Y_test, Y_pred))
    precision_scores.append(precision_score(Y_test, Y_pred, zero_division=1))
    recall_scores.append(recall_score(Y_test, Y_pred, zero_division=1))
    cm_scores.append(confusion_matrix(Y_test, Y_pred))
    
    acc = accuracy_score(Y_test, Y_pred)
    print(f'Fold {k} - Accuracy-score: {round(acc, 2)} - Best Parameters: {best_parameters}')

baseline_model_acc = accuracy_scores
baseline_model_param = best_parameters_list[np.argmax(accuracy_scores)]

print(' ')
print('Mean Accuracy-score:', f'{round(np.mean(accuracy_scores), 2)}.', 'Std Accuracy-score:', f'{round(np.std(accuracy_scores), 2)}.', 'Max Accuracy-score:', round(np.max(accuracy_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(accuracy_scores)]}')
print(' ')
print('Mean F1-score:', f'{round(np.mean(f1_scores), 2)}.', 'Std F1-score:', f'{round(np.std(f1_scores), 2)}.','Max F1-score:', round(np.max(f1_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(f1_scores)]}')
print(' ')
print('Mean Precision-score:', f'{round(np.mean(precision_scores), 2)}.', 'Std Precision-score:', f'{round(np.std(precision_scores), 2)}.', 'Max Precision-score:', round(np.max(precision_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(precision_scores)]}')
print(' ')
print('Mean Recall-score:', f'{round(np.mean(recall_scores), 2)}.', 'Std Recall-score:', f'{round(np.std(recall_scores), 2)}.', 'Max Recall-score:', round(np.max(recall_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(recall_scores)]}')
print(' ')

cm_plot(cm_scores, accuracy_scores, model='Baseline', total=True)

In [ ]:
# Print features selected by the model

indexes_baseline = np.where(selected_feature_indices)[0]
node_names_orig = ['F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'Cz', 'T8', 'P3', 'Pz', 'P4', 'PO7', 'PO8', 'Oz']
combinations_total = node_names_orig*3

alpha_spec, beta_spec, theta_spec = [], [], []
for i in indexes_baseline:
    if i < 14:
        alpha_spec.append(combinations_total[i])
    elif i >= 14 and i < 28:
        beta_spec.append(combinations_total[i])
    elif i >= 28 and i < 42:
        theta_spec.append(combinations_total[i])
print(f'Number of chosen spectral features per frequency band: , \nAlpha: {len(alpha_spec)} \nBeta: {len(beta_spec)} \nTheta: {len(theta_spec)}')
print(f'Spectral features per frequency band: \nAlpha: {alpha_spec} \nBeta: {beta_spec} \nTheta: {theta_spec}')

chosen_frequency_bands = [alpha_spec, beta_spec, theta_spec]
  

# Model PLV - RFE on 42 spectral features + 30 frontal-parietal PLV features and SVM

In [ ]:
# Loading data 

directory = '/Flight-Sim-Cognitive-Workload-EEG-Prediction/data'
concatenated_X, concatenated_Y = [], []
for subdir in sorted(os.listdir(directory)):
    for file in sorted(os.listdir(os.path.join(directory, subdir))):
        print(directory.split('/')[-1], subdir, file)
        data = np.load(os.path.join(directory, subdir, file))
        X_PLV = fronto_parietal_averagePLV(data['X'][:135]) # Output: 30 PLV features (10 alpha, 10 beta, 10 theta)
        X_Freq = data['X'][135:] # frequency spectrum features
        X = np.concatenate((X_PLV, X_Freq))
        Y = data['Y']
        concatenated_X.append(X)
        concatenated_Y.append(Y)

concatenated_Y = np.concatenate(concatenated_Y)


In [ ]:
# Recursive Feature Selection and Support Vector Machine - Binary Classification (RFE + SVM)

X = concatenated_X
Y = concatenated_Y
classifier = svm.SVC(random_state=42)
parameters = {'C': [0.1, 0.95, 0.99, 10], 'kernel': ['linear', 'rbf', 'sigmoid', 'poly'], 'gamma': [0.0001, 0.01, 0.95, 1, 10, 25]}
scoring = make_scorer(accuracy_score)
num_folds = 10
num_features = 8
kf = KFold(n_splits=num_folds)
f1_scores, accuracy_scores, precision_scores, recall_scores, cm_scores, best_parameters_list = [],[],[],[],[],[]

for k, (train_index, test_index) in enumerate(kf.split(X), start=1):
    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    Y_train, Y_test = Y[train_index], Y[test_index]
    
    # Perform feature selection using RFE with LinearSVC as the estimator
    estimator = svm.LinearSVC(max_iter=1000000)
    selector = RFE(estimator, n_features_to_select=num_features)
    selector.fit(X_train, Y_train)
    X_train_selected = selector.transform(X_train)
    X_test_selected = selector.transform(X_test)
    
    selected_feature_indices = selector.support_
    selected_feature_ranking = selector.ranking_

    grid_search = GridSearchCV(classifier, parameters, scoring=scoring)
    grid_search.fit(X_train_selected, Y_train)

    best_parameters = grid_search.best_params_

    classifier = svm.SVC(kernel=best_parameters['kernel'], C=best_parameters['C'], gamma=best_parameters['gamma'])
    classifier.fit(X_train_selected, Y_train)

    Y_pred = classifier.predict(X_test_selected)

    best_parameters_list.append(best_parameters) 
    f1_scores.append(f1_score(Y_test, Y_pred, zero_division=1))
    accuracy_scores.append(accuracy_score(Y_test, Y_pred))
    precision_scores.append(precision_score(Y_test, Y_pred, zero_division=1))
    recall_scores.append(recall_score(Y_test, Y_pred, zero_division=1))
    cm_scores.append(confusion_matrix(Y_test, Y_pred))
    
    acc = accuracy_score(Y_test, Y_pred)
    print(f'Fold {k} - Accuracy-score: {round(acc, 2)} - Best Parameters: {best_parameters}')

model_PLV_acc = accuracy_scores
model_PLV_param = best_parameters_list[np.argmax(accuracy_scores)]

print(' ')
print('Mean Accuracy-score:', f'{round(np.mean(accuracy_scores), 2)}.', 'Std Accuracy-score:', f'{round(np.std(accuracy_scores), 2)}.', 'Max Accuracy-score:', round(np.max(accuracy_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(accuracy_scores)]}')
print(' ')
print('Mean F1-score:', f'{round(np.mean(f1_scores), 2)}.', 'Std F1-score:', f'{round(np.std(f1_scores), 2)}.','Max F1-score:', round(np.max(f1_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(f1_scores)]}')
print(' ')
print('Mean Precision-score:', f'{round(np.mean(precision_scores), 2)}.', 'Std Precision-score:', f'{round(np.std(precision_scores), 2)}.', 'Max Precision-score:', round(np.max(precision_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(precision_scores)]}')
print(' ')
print('Mean Recall-score:', f'{round(np.mean(recall_scores), 2)}.', 'Std Recall-score:', f'{round(np.std(recall_scores), 2)}.', 'Max Recall-score:', round(np.max(recall_scores), 2), '\n', f'with Hyperparameter set: {best_parameters_list[np.argmax(recall_scores)]}')
print(' ')

cm_plot(cm_scores, accuracy_scores, model='Connectivity_Model', total=True)

In [ ]:
# Print spectral and PLV features selected by the model

indexes = np.where(selected_feature_indices)[0]
plv_names = ['F7', 'F3', 'Fz', 'F4', 'F8', 'P3', 'Pz', 'P4', 'PO7', 'PO8']
node_names_orig = ['F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'Cz', 'T8', 'P3', 'Pz', 'P4', 'PO7', 'PO8', 'Oz']
combinations_total = plv_names*3 + node_names_orig*3

alpha_plv, beta_plv, theta_plv = [], [], []
alpha_spec, beta_spec, theta_spec = [], [], []
for i in indexes:
    if i < 10:
        alpha_plv.append(combinations_total[i])
    elif i >= 10 and i < 20:
        beta_plv.append(combinations_total[i])
    elif i >= 20 and i < 30:
        theta_plv.append(combinations_total[i])
    elif i >= 30 and i < 44:
        alpha_spec.append(combinations_total[i])
    elif i >= 44 and i < 58:
        beta_spec.append(combinations_total[i])
    elif i >= 58 and i < 72:
        theta_spec.append(combinations_total[i])
        
print('Number of PLV features chosen: ', len(alpha_plv), len(beta_plv), len(theta_plv))
print(f'PLV features per frequency band: \nAlpha: {alpha_plv} \nBeta: {beta_plv} \nTheta: {theta_plv}')
print(' ')
print('Number of Spectral features chosen: ', len(alpha_spec), len(beta_spec), len(theta_spec))
print(f'Spectral features per frequency band: \nAlpha: {alpha_spec} \nBeta: {beta_spec} \nTheta: {theta_spec}')

chosen_PLVs = [alpha_plv, beta_plv, theta_plv]
modelPLV_chosen_frequency_bands = [alpha_spec, beta_spec, theta_spec]

# Box plot and Results summary

In [ ]:
# Boxplot accuracy-scores per model.

data = [baseline_model_acc, model_PLV_acc]
sns.set(style="white")
palette = ["lightblue"]

plt.figure(figsize=(8, 6))
sns.boxplot(data=data, palette=palette, linewidth=2, boxprops={"edgecolor": "darkblue"}, medianprops={"color": "darkblue"}, whiskerprops={"color": "darkblue"}, capprops={"color": "darkblue"})
plt.xticks(ticks=range(len(['Baseline Model', 'Connectivity Model'])),labels=['Baseline Model', 'Connectivity Model'])
plt.ylabel("Accuracy")
plt.ylim([0,1])

# Add median scores on the y-axis
medians = [np.median(baseline_model_acc), np.median(model_PLV_acc)]
for i, median in enumerate(medians):
    plt.text(i, median, f'{median:.2f}', horizontalalignment='center', verticalalignment='bottom', fontdict={'weight': 'bold', 'size': 12}, color='darkblue')

sns.despine()
plt.show()

output_file = '/Flight-Sim-Cognitive-Workload-EEG-Prediction/results/boxplot_accuracy.png'
plt.savefig(output_file, bbox_inches='tight')
plt.clf()


In [ ]:
# Selected spectral features and PLV features per model.

chosen_FREQUENCYBANDs = chosen_frequency_bands, modelPLV_chosen_frequency_bands
models = ['Baseline Model', 'Connectivity Model']
for name, selected in zip(models, chosen_FREQUENCYBANDs):
    print(f'Selected spectral features of {name}: {selected}')

name = 'Connectivity Model'
print(f'Selected Connectivity (PLV) features of {name}: {chosen_PLVs}') 

In [ ]:
# Print model with highest accuracy among folds

acc_scorers = baseline_model_acc, model_PLV_acc
chosen_param = baseline_model_param, model_PLV_param

new_list = []
for i in acc_scorers:
    idx = np.argmax(i)
    new_list.append(i[idx])
idx1 = np.argmax(new_list)

if idx1 == 0:
    model = 'Baseline Model' 
elif idx1 == 1:
    model = 'Connectivity Model'
print(f'Best performing model based on highest accuracy among folds: {model} \n max accuracy score: {new_list[idx1]}. \n mean accuracy score: {np.mean(acc_scorers[idx1])}. \n best performing set of hyperparameters: {chosen_param[idx1]}.')

In [ ]:
# Print best performing model based on highest mean accuracy over folds.

mean_acc_scorers = np.mean(acc_scorers, axis=1)
idx = np.argmax(mean_acc_scorers)
idx1 = np.argmax(acc_scorers[idx])
if idx == 0:
    model = 'Baseline Model'
elif idx == 1:
    model = 'Connectivity Model'
print(f'Best performing model based on highest mean accuracy: {model} \n max accuracy score: {acc_scorers[idx][idx1]}. \n mean accuracy score: {np.mean(mean_acc_scorers[idx])}. \n best performing set of hyperparameters: {chosen_param[idx]}.')